# CDCR Hazard Heatmap Table — All 30 Facilities

One row per active CDCR state prison. Four hazard columns (heat, drought, flood, wildfire).  
Cell values show human-readable components; Jenks natural breaks (3-class) on the underlying index drive the heatmap classification.

**Outputs:** Two CSVs (alphabetical sort, multi-hazard composite score sort) with `_level` columns for manual color application.

**Color legend (sequential warm, colorblind-safe):**
| Level | Hex | Usage |
|-------|-----|-------|
| Higher | `#D73027` | Heat/drought/flood top Jenks class |
| Medium | `#FC8D59` | Heat/drought/flood middle class |
| Lower | `#FEE08B` | Heat/drought/flood bottom class |
| Very High | `#D73027` | Wildfire FHSZ |
| High | `#FC8D59` | Wildfire FHSZ |
| Moderate | `#FEE08B` | Wildfire FHSZ |
| None | `#FFFFFF` | Wildfire FHSZ / no hazard |

In [ ]:
import pandas as pd
import numpy as np
import jenkspy
from pathlib import Path

ROOT = Path('../../')
DATA = ROOT / 'data'
DATA_SRC = ROOT / 'data_sources'

# Load multi-hazard rank (has indices, scores, and most components)
mhr = pd.read_csv('CDCR_multi_hazard_rank.csv')

# Load sources for missing components: drought summer temp, heat days over 80F
cdcr = pd.read_csv(DATA / 'cdcr' / 'cdcr_facilities.csv')
drought_tracts = pd.read_csv(DATA / 'hazards' / 'drought_hazard.csv')
heat_tracts = pd.read_csv(DATA_SRC / 'hazards' / 'heat' / 'heatdays_alltimes_tract.csv')

print(f'Multi-hazard rank rows: {len(mhr)}')
print(f'Columns: {list(mhr.columns)}')

In [ ]:
# Join missing component columns via tract GEOID
cdcr_tracts = cdcr[['cdcr_code', 'tract_geoid']].dropna(subset=['cdcr_code'])
cdcr_tracts['tract_geoid'] = cdcr_tracts['tract_geoid'].astype(str).str.zfill(11)
drought_tracts['GEOID'] = drought_tracts['GEOID'].astype(str).str.zfill(11)
heat_tracts['GEOID'] = heat_tracts['GEOID'].astype(str).str.zfill(11)

df = mhr.merge(cdcr_tracts, on='cdcr_code', how='left')

df = df.merge(
    drought_tracts[['GEOID', 'Dr_delta_JA_max_fut']],
    left_on='tract_geoid', right_on='GEOID', how='left'
).drop(columns='GEOID')

df = df.merge(
    heat_tracts[['GEOID', 'days_over_80_historic', 'days_over_80_midcentury']],
    left_on='tract_geoid', right_on='GEOID', how='left'
).drop(columns='GEOID')

# Compute heat % change (80F days)
df['heat_80f_pct_change'] = (
    (df['days_over_80_midcentury'] - df['days_over_80_historic'])
    / df['days_over_80_historic'] * 100
)

print(f'Joined rows: {len(df)}')
print(f'Nulls in new columns:')
print(df[['Dr_delta_JA_max_fut', 'days_over_80_midcentury', 'heat_80f_pct_change']].isnull().sum())

In [ ]:
# Clean facility names
name_fixes = {
    "Central California Women'S Facility": "Central California Women's Facility",
    "California Men'S Colony": "California Men's Colony",
    "Ca Substance Abuse Treatment Facility": "CA Substance Abuse Treatment Facility",
}
df['facility_name'] = df['facility_name'].replace(name_fixes)

# Build facility label: "Name (CODE)\nCity, CA"
df['facility_label'] = (
    df['facility_name'] + ' (' + df['cdcr_code'] + ')\n' + df['city'] + ', CA'
)

print('Sample labels:')
for label in df['facility_label'].head(5):
    print(repr(label))

In [ ]:
# Jenks natural breaks (3 classes) for heat, drought, flood indices
heat_breaks = jenkspy.jenks_breaks(df['heat_raw'].dropna().values, n_classes=3)
drought_breaks = jenkspy.jenks_breaks(df['drought_raw'].dropna().values, n_classes=3)
flood_breaks = jenkspy.jenks_breaks(df['flood_raw'].dropna().values, n_classes=3)

def classify_jenks(val, breaks, labels=('Lower', 'Medium', 'Higher')):
    """Classify a value into Jenks break classes."""
    if pd.isna(val):
        return None
    for i in range(1, len(breaks)):
        if val <= breaks[i]:
            return labels[i - 1]
    return labels[-1]

df['heat_level'] = df['heat_raw'].apply(lambda x: classify_jenks(x, heat_breaks))
df['drought_level'] = df['drought_raw'].apply(lambda x: classify_jenks(x, drought_breaks))
df['flood_level'] = df['flood_raw'].apply(lambda x: classify_jenks(x, flood_breaks))
df['wildfire_level'] = df['wildfire_fhsz'].fillna('None')

print('=== Jenks break points ===')
print(f'Heat index:    {[round(b, 1) for b in heat_breaks]}')
print(f'Drought index: {[round(b, 1) for b in drought_breaks]}')
print(f'Flood index:   {[round(b, 1) for b in flood_breaks]}')

print('\n=== Classification counts ===')
for col in ['heat_level', 'drought_level', 'flood_level', 'wildfire_level']:
    print(f'\n{col}:')
    print(df[col].value_counts())

In [ ]:
# Format cell values (components only, no parentheses)

# Heat: "210 days\n+24% increase"
df['heat_cell'] = df.apply(
    lambda r: f"{r['days_over_80_midcentury']:.0f} days\n+{r['heat_80f_pct_change']:.0f}% increase",
    axis=1
)

# Drought: "'+10.3F summer temp\n58.3 WSV" (leading ' prevents Sheets equation parse)
df['drought_cell'] = df.apply(
    lambda r: f"'+{r['Dr_delta_JA_max_fut'] * 9/5:.1f}F summer temp\n{r['drought_wsv']:.1f} WSV",
    axis=1
)

# Flood: "100.0% in floodplain\n21.3% very wet days"
df['flood_cell'] = df.apply(
    lambda r: f"{r['flood_500yr_floodplain_pct']:.1f}% in floodplain\n{r['flood_verywet_pct']:.1f}% very wet days",
    axis=1
)

# Wildfire: FHSZ category
df['wildfire_cell'] = df['wildfire_fhsz'].fillna('None')

# Preview
print('Sample cells:')
for _, r in df.head(3).iterrows():
    print(f"\n--- {r['cdcr_code']} ---")
    print(f"Heat: {repr(r['heat_cell'])}")
    print(f"Drought: {repr(r['drought_cell'])}")
    print(f"Flood: {repr(r['flood_cell'])}")
    print(f"Wildfire: {r['wildfire_cell']}")

In [ ]:
# Build output columns
out_cols = {
    'facility_label': 'Facility',
    'heat_cell': 'Heat (Days Over 80F Mid-Century)',
    'drought_cell': 'Drought (Mid-Century Index)',
    'flood_cell': 'Flood (Mid-Century Index)',
    'wildfire_cell': 'Wildfire (FHSZ)',
    'heat_level': 'heat_level',
    'drought_level': 'drought_level',
    'flood_level': 'flood_level',
    'wildfire_level': 'wildfire_level',
}

# Add top-10 indicator columns (method='min' so ties at boundary are included)
df['heat_top10'] = df['heat_raw'].rank(ascending=False, method='min') <= 10
df['drought_top10'] = df['drought_raw'].rank(ascending=False, method='min') <= 10
df['flood_top10'] = df['flood_raw'].rank(ascending=False, method='min') <= 10
# Wildfire: top 10 by FHSZ ordinal (Very High=3, High=2, Moderate=1, None=0)
fhsz_ord = {'Very High': 3, 'High': 2, 'Moderate': 1}
df['fire_ord'] = df['wildfire_fhsz'].map(fhsz_ord).fillna(0)
df['wildfire_top10'] = df['fire_ord'].rank(ascending=False, method='min') <= 10

out_cols['heat_top10'] = 'heat_top10'
out_cols['drought_top10'] = 'drought_top10'
out_cols['flood_top10'] = 'flood_top10'
out_cols['wildfire_top10'] = 'wildfire_top10'

# === Alphabetical sort ===
alpha = df.sort_values('facility_name').reset_index(drop=True)
alpha_out = alpha[list(out_cols.keys())].rename(columns=out_cols)
alpha_out.to_csv('CDCR_hazard_heatmap_alpha.csv', index=False)
print(f'Saved CDCR_hazard_heatmap_alpha.csv ({len(alpha_out)} rows)')

# === Multi-hazard composite score sort ===
score = df.sort_values('multi_hazard_score', ascending=False).reset_index(drop=True)
score_out = score[list(out_cols.keys())].rename(columns=out_cols)
score_out.to_csv('CDCR_hazard_heatmap_score.csv', index=False)
print(f'Saved CDCR_hazard_heatmap_score.csv ({len(score_out)} rows)')

# Verify top-10 counts (may be >10 due to ties)
print(f"\nTop-10 counts: heat={df['heat_top10'].sum()}, drought={df['drought_top10'].sum()}, flood={df['flood_top10'].sum()}, wildfire={df['wildfire_top10'].sum()}")

print('\n=== Color legend ===')
print('Higher / Very High:  #D73027 (dark red)')
print('Medium / High:       #FC8D59 (orange)')
print('Lower / Moderate:    #FEE08B (light yellow)')
print('None:                #FFFFFF (white)')

In [ ]:
# === Points-based sort (top-10 heat/drought/flood by days over 80F/index, top-5 wildfire) ===
# 1 point per hazard if facility is in top 10 (top 5 for wildfire)
df['heat_top10_80f'] = df['days_over_80_midcentury'].rank(ascending=False, method='min') <= 10
df['drought_top10'] = df['drought_raw'].rank(ascending=False, method='min') <= 10
df['flood_top10'] = df['flood_raw'].rank(ascending=False, method='min') <= 10
fhsz_ord = {'Very High': 3, 'High': 2, 'Moderate': 1}
df['fire_ord'] = df['wildfire_fhsz'].map(fhsz_ord).fillna(0)
df['wildfire_top5'] = df['fire_ord'].rank(ascending=False, method='min') <= 5

df['points'] = (df['heat_top10_80f'].astype(int) + df['drought_top10'].astype(int)
                + df['flood_top10'].astype(int) + df['wildfire_top5'].astype(int))

points_out_cols = {
    'facility_label': 'Facility',
    'heat_cell': 'Heat (Days Over 80F Mid-Century)',
    'drought_cell': 'Drought (Mid-Century Index)',
    'flood_cell': 'Flood (Mid-Century Index)',
    'wildfire_cell': 'Wildfire (FHSZ)',
    'heat_top10_80f': 'heat_bold',
    'drought_top10': 'drought_bold',
    'flood_top10': 'flood_bold',
    'wildfire_top5': 'wildfire_bold',
    'points': 'points',
}

points_df = df.sort_values(['points', 'multi_hazard_score'], ascending=[False, False]).reset_index(drop=True)
points_out = points_df[list(points_out_cols.keys())].rename(columns=points_out_cols)
points_out.to_csv('CDCR_hazard_heatmap_points.csv', index=False)
print(f'Saved CDCR_hazard_heatmap_points.csv ({len(points_out)} rows)')
print(f"Points distribution: {points_df['points'].value_counts().sort_index(ascending=False).to_dict()}")